In [1]:
"""Train the ECOD detector and scale its anomaly scores."""

import os
import sys

import pandas as pd
from pyod.models.ecod import ECOD
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
import config
import to_tflite

# Build the detector from its configuration.
ecod = ECOD(
    contamination=config.CONTAMINATION,
    n_jobs=config.N_JOBS,
)

# NOTE: all dataset features must be pre-normalized so that, after the int8
# quantization in keras/to_tflite.py, no value overflows the int8 range.
# Fit the configured features from the CSV.
features = pd.read_csv(config.CSV_PATH)[config.FEATURES].to_numpy()
X_train, X_test = train_test_split(
    features, test_size=config.TEST_SIZE, random_state=config.RANDOM_STATE
)
ecod.fit(X_train)

# Scale the raw anomaly scores into the configured range.
raw_scores = ecod.decision_function(X_train).reshape(-1, 1)
score_scaler = MinMaxScaler(feature_range=config.SCORE_RANGE).fit(raw_scores)
train_scores = score_scaler.transform(raw_scores).ravel()
test_scores = score_scaler.transform(
    ecod.decision_function(X_test).reshape(-1, 1)
).ravel()

# Learn the scores with a small network and export .keras + .tflite.
to_tflite.to_tflite(
    name="ecod",
    x_train=X_train,
    y_train=train_scores,
    x_test=X_test,
    y_test=test_scores,
    layers=config.LAYERS,
    learning_rate=config.LEARNING_RATE,
    epochs=config.EPOCHS,
    batch_size=config.BATCH_SIZE,
    verbose=config.VERBOSE,
    feature_names=config.FEATURES,
)


2026-08-14 18:30:30.658777: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


Epoch 1/50


2026-08-14 18:30:33.611364: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


3747/3747 ━━━━━━━━━━━━━━━━━━━━ 8s 2ms/step - loss: 9.6330e-04 - val_loss: 4.7779e-04
Epoch 2/50
3747/3747 ━━━━━━━━━━━━━━━━━━━━ 10s 3ms/step - loss: 4.0930e-04 - val_loss: 3.2951e-04
Epoch 3/50
3747/3747 ━━━━━━━━━━━━━━━━━━━━ 8s 2ms/step - loss: 2.4293e-04 - val_loss: 2.2911e-04
Epoch 4/50
3747/3747 ━━━━━━━━━━━━━━━━━━━━ 14s 3ms/step - loss: 2.1994e-04 - val_loss: 2.1697e-04
Epoch 5/50
3747/3747 ━━━━━━━━━━━━━━━━━━━━ 16s 2ms/step - loss: 2.1945e-04 - val_loss: 2.2861e-04
Epoch 6/50
3747/3747 ━━━━━━━━━━━━━━━━━━━━ 10s 3ms/step - loss: 2.1848e-04 - val_loss: 2.0847e-04
Epoch 7/50
3747/3747 ━━━━━━━━━━━━━━━━━━━━ 8s 2ms/step - loss: 2.1580e-04 - val_loss: 2.2172e-04
Epoch 8/50
3747/3747 ━━━━━━━━━━━━━━━━━━━━ 7s 2ms/step - loss: 2.1925e-04 - val_loss: 2.2246e-04
Epoch 9/50
3747/3747 ━━━━━━━━━━━━━━━━━━━━ 10s 3ms/step - loss: 2.1389e-04 - val_loss: 2.0067e-04
Epoch 10/50
3747/3747 ━━━━━━━━━━━━━━━━━━━━ 10s 3ms/step - loss: 2.0995e-04 - val_loss: 2.1458e-04
Epoch 11/50
3747/3747 ━━━━━━━━━━━━━━━━━━━━ 7

INFO:tensorflow:Assets written to: /tmp/tmpnb1n7a37/assets


Saved artifact at '/tmp/tmpnb1n7a37'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 2), dtype=tf.float32, name='keras_tensor')
Output Type:
  TensorSpec(shape=(None, 1), dtype=tf.float32, name=None)
Captures:
  139940157256912: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139940157259600: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139940153768464: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139940153768848: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139940153768272: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139940153767120: TensorSpec(shape=(), dtype=tf.resource, name=None)


/home/patrik/Diplomka/LODA/env/lib/python3.12/site-packages/tensorflow/lite/python/convert.py:863: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(
W0000 00:00:1786725435.979597   24108 tf_tfl_flatbuffer_helpers.cc:364] Ignored output_format.
W0000 00:00:1786725435.979613   24108 tf_tfl_flatbuffer_helpers.cc:367] Ignored drop_control_dependency.
2026-08-14 18:37:15.979823: I tensorflow/cc/saved_model/reader.cc:83] Reading SavedModel from: /tmp/tmpnb1n7a37
2026-08-14 18:37:15.980083: I tensorflow/cc/saved_model/reader.cc:52] Reading meta graph with tags { serve }
2026-08-14 18:37:15.980087: I tensorflow/cc/saved_model/reader.cc:147] Reading SavedModel debug info (if present) from: /tmp/tmpnb1n7a37
I0000 00:00:1786725435.986867   24108 mlir_graph_optimization_pass.cc:437] MLIR V1 optimization pass is not enabled
2026-08-14 18:37:15.987209: I tensorflow/cc/saved_model/loader.cc:236] Restoring SavedModel bundle.
2026-08-14 1

<Sequential name=sequential, built=True>